# NaijaReviewer-8B — End-to-End Pipeline (Drive-Persisted, Resume-Safe)

**One notebook. Five parts. Drive-backed. Auto-resumes after disconnects. GPU-autotuned.**

| Part | What | LLM / GPU | Time | Cost |
|---|---|---|---|---|
| 1 | Build ~20k Nigerian fine-tune corpus | NVIDIA Nemotron + Claude | ~1 hr | ~$2–5 |
| 2 | QLoRA fine-tune Llama 3.1 8B → NaijaReviewer-8B | local A100/L4/H100 | ~1.5–3 hr | $0 |
| 3 | Head-to-head eval vs. baselines | Nemotron + Claude + local | ~20 min | ~$1 |
| 4 | Push model + dataset to HuggingFace | — | ~5 min | $0 |
| 5 | GGUF conversion notes for Ollama deploy | — | local | $0 |

**Inputs needed:**
- `NVIDIA_API_KEY` — from build.nvidia.com (Nemotron synthetic + rating refinement)
- `ANTHROPIC_API_KEY` — for AfriSenti Pidgin reformulation
- `HF_TOKEN` — for model + dataset push (also needed for Llama 3.1 8B download — accept the license first at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct)

## Why Drive-persisted?

Colab Free / Pro disconnects after **~12 hours**; idle disconnects after **~90 minutes**. A 2-hour fine-tune run is right at the edge. Every byte we write goes to Drive, so re-running the notebook picks up exactly where it left off — corpus stages, training checkpoints, merged model, eval results.

## Resumability

**Just re-run the notebook from the top.** Each stage:
- Skips if its output file already exists in Drive
- Training: detects existing `checkpoint-XXXX` and resumes from it
- HF push: skips files that already match remote hash

Architecture follows PRD v4 (`PRD/PRD_v4_NPA_5Day.md`).

## 0. Setup — Drive + Repo + Deps + Secrets + GPU Autotune

In [ ]:
# Mount Google Drive — everything writeable lives here so disconnects don't lose work
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_DIR = Path("/content/drive/MyDrive/naija-persona-agent")
else:
    DRIVE_DIR = Path.home() / "naija-persona-agent-runs"

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"✅ Persistent working dir: {DRIVE_DIR}")

# Subdirs we'll use everywhere
CORPUS_DIR = DRIVE_DIR / "data" / "finetune"
CHECKPOINT_DIR = DRIVE_DIR / "checkpoints" / "naija-reviewer-v1"
MERGED_DIR = DRIVE_DIR / "merged" / "naija-reviewer-8b"
RESULTS_DIR = DRIVE_DIR / "results"
for d in (CORPUS_DIR, CHECKPOINT_DIR, MERGED_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"  corpus:      {CORPUS_DIR}")
print(f"  checkpoints: {CHECKPOINT_DIR}")
print(f"  merged:      {MERGED_DIR}")
print(f"  results:     {RESULTS_DIR}")

In [ ]:
# Clone the repo (Colab) or chdir to it (local)
REPO_URL = "https://github.com/Mystique1337/telcoproject"

if IN_COLAB:
    REPO_DIR = Path("/content/telcoproject")
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        # Pull latest if already cloned
        !cd {REPO_DIR} && git pull --rebase --autostash || true
    %cd {REPO_DIR}
else:
    if not (Path.cwd() / "app").exists():
        for candidate in [Path.cwd().parent, Path.cwd() / "telcoproject"]:
            if (candidate / "app").exists():
                os.chdir(candidate)
                break
    REPO_DIR = Path.cwd()

assert (Path.cwd() / "app").exists(), "Couldn't locate repo root"

# Symlink the local data/finetune to Drive so the corpus builder script writes to Drive
local_finetune = REPO_DIR / "data" / "finetune"
if local_finetune.is_symlink():
    local_finetune.unlink()
elif local_finetune.exists() and not any(local_finetune.iterdir()):
    local_finetune.rmdir()
if not local_finetune.exists():
    local_finetune.symlink_to(CORPUS_DIR)
    print(f"  ✅ symlinked {local_finetune} → {CORPUS_DIR}")
else:
    print(f"  ⚠️ {local_finetune} exists and is not empty; leaving as-is")

print(f"  Repo: {REPO_DIR}")

In [ ]:
# Dependencies — split into fast (data) and slow (training) groups so re-runs are quick
%pip install -q \
    httpx==0.27.* anthropic==0.39.* datasets==3.1.* \
    pyyaml tqdm nest_asyncio \
    pydantic==2.9.* pydantic-settings==2.6.* jinja2 chromadb==0.5.*

# Training deps — heavier; takes ~2 min
%pip install -q \
    "torch>=2.5" "transformers>=4.46" "peft>=0.13" "accelerate>=1.1" \
    "trl>=0.12" "bitsandbytes>=0.44" wandb sentencepiece protobuf

# Try Flash Attention 2 — big speedup on A100/H100, optional. Comment out if it errors.
%pip install -q flash-attn --no-build-isolation 2>/dev/null || echo "flash-attn install skipped (will fall back to SDPA)"

print("\n✅ Deps installed")

In [ ]:
# Secrets — Colab Secrets (sidebar key icon) preferred; falls back to .env / prompt
def _set_secret(name: str, prompt: str) -> str:
    if IN_COLAB:
        from google.colab import userdata
        try:
            v = userdata.get(name)
            if v:
                os.environ[name] = v
                print(f"  ✅ {name} loaded from Colab Secrets")
                return v
        except Exception:
            pass
    v = os.environ.get(name)
    if not v:
        try:
            from dotenv import load_dotenv; load_dotenv()
            v = os.environ.get(name)
        except ImportError:
            pass
    if not v:
        from getpass import getpass
        v = getpass(prompt)
        os.environ[name] = v
    print(f"  ✅ {name} configured")
    return v

_set_secret("NVIDIA_API_KEY",   "NVIDIA_API_KEY (nvapi-…): ")
_set_secret("ANTHROPIC_API_KEY","ANTHROPIC_API_KEY (sk-ant-…): ")
_set_secret("HF_TOKEN",         "HF_TOKEN (hf_…): ")

In [ ]:
# Import app helpers + apply nest_asyncio for async-in-notebook
sys.path.insert(0, str(REPO_DIR))

import nest_asyncio
nest_asyncio.apply()

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s — %(message)s")

from app.config import get_settings
from app.llm.client import LLMClient, LLMError

settings = get_settings()
print(f"NVIDIA endpoint:    {settings.nvidia_base_url}")
print(f"NVIDIA key set:     {bool(settings.nvidia_api_key)}")
print(f"Anthropic key set:  {bool(settings.anthropic_api_key)}")
print(f"HF token set:       {bool(os.environ.get('HF_TOKEN'))}")

In [ ]:
# GPU autotune — detects the GPU type and picks optimal training hyperparams
import torch

def gpu_autotune():
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU — Runtime → Change runtime type → A100/L4/H100")
    name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    bf16_ok = torch.cuda.is_bf16_supported()
    flash_attn_ok = False
    try:
        import flash_attn  # noqa: F401
        flash_attn_ok = True
    except ImportError:
        pass

    # Decision matrix — effective batch size target = 32 for all GPUs
    name_l = name.lower()
    if "h100" in name_l:
        cfg = {"batch": 16, "grad_accum": 2, "attn": "flash_attention_2" if flash_attn_ok else "sdpa",
               "workers": 4, "seq_len": 4096}
    elif "a100" in name_l and vram_gb > 60:
        cfg = {"batch": 8, "grad_accum": 4, "attn": "flash_attention_2" if flash_attn_ok else "sdpa",
               "workers": 4, "seq_len": 4096}
    elif "a100" in name_l:
        cfg = {"batch": 4, "grad_accum": 8, "attn": "flash_attention_2" if flash_attn_ok else "sdpa",
               "workers": 4, "seq_len": 4096}
    elif "l4" in name_l:
        cfg = {"batch": 2, "grad_accum": 16, "attn": "sdpa",
               "workers": 2, "seq_len": 3072}
    elif "t4" in name_l:
        cfg = {"batch": 1, "grad_accum": 32, "attn": "sdpa",
               "workers": 2, "seq_len": 2048}
        if not bf16_ok:
            print("  ⚠️ T4 doesn't support bf16 — training will use fp16 (slightly less stable)")
    else:
        cfg = {"batch": 2, "grad_accum": 16, "attn": "sdpa", "workers": 2, "seq_len": 3072}
    return name, vram_gb, bf16_ok, flash_attn_ok, cfg

GPU_NAME, GPU_VRAM, BF16_OK, FLASH_OK, GPU_CFG = gpu_autotune()

# Tensorfloat-32 — free 2-3× speedup on A100/H100 matmul for bf16 training
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print(f"GPU:            {GPU_NAME}")
print(f"VRAM:           {GPU_VRAM:.1f} GB")
print(f"bf16 supported: {BF16_OK}")
print(f"Flash-Attn 2:   {FLASH_OK}")
print(f"Autotuned config:")
print(f"  batch_size:       {GPU_CFG['batch']}")
print(f"  grad_accum:       {GPU_CFG['grad_accum']}")
print(f"  effective batch:  {GPU_CFG['batch'] * GPU_CFG['grad_accum']}")
print(f"  attention impl:   {GPU_CFG['attn']}")
print(f"  DataLoader workers: {GPU_CFG['workers']}")
print(f"  max seq length:   {GPU_CFG['seq_len']}")

In [ ]:
# Connectivity sanity check — burns ~50 tokens
import asyncio

async def _ping():
    nv = LLMClient("nvidia:nvidia/llama-3.1-nemotron-70b-instruct")
    cl = LLMClient("anthropic:claude-sonnet-4-20250514")
    r1 = await nv.complete("Say 'NVIDIA OK'.", max_tokens=10, temperature=0)
    r2 = await cl.complete("Say 'CLAUDE OK'.", max_tokens=10, temperature=0)
    return r1.strip(), r2.strip()

try:
    nv_ok, cl_ok = asyncio.run(_ping())
    print(f"  Nemotron: {nv_ok}")
    print(f"  Claude:   {cl_ok}")
    print("\n✅ Both endpoints reachable")
except Exception as e:
    print(f"❌ Endpoint error: {e}")

## Part 1 — Build the Fine-Tune Corpus

Six stages, all output written to **Drive** (`CORPUS_DIR`):

1. Download `aymane-maghouti/Sentiment-Analysis-for-Jumia-Reviews` → 15k stratified subsample
2. **Nemotron**: binary sentiment → 1–5 star ratings + primary aspect
3. **Claude**: AfriSenti pcm → reframed as Pidgin product reviews
4. **Nemotron**: persona × product × register × rating synthetic generation
5. Combine + dedup + stratified 90/5/5 split
6. Write `DATA_CARD.md`

**Resumable**: each stage skips if its output JSONL already exists on Drive. Safe to re-run.

In [ ]:
# Check what already exists on Drive (resume signal)
print("=== Existing corpus stage outputs on Drive ===")
for stage_file in sorted((CORPUS_DIR / "stages").glob("*.jsonl") if (CORPUS_DIR / "stages").exists() else []):
    n = sum(1 for _ in stage_file.open())
    print(f"  ✅ {stage_file.name:50s} {n:6d} rows ({stage_file.stat().st_size/1e6:.1f} MB)")

for final in ["v1_train.jsonl", "v1_val.jsonl", "v1_test.jsonl", "DATA_CARD.md"]:
    p = CORPUS_DIR / final
    if p.exists():
        if p.suffix == ".jsonl":
            n = sum(1 for _ in p.open())
            print(f"  ✅ FINAL  {final:25s}  {n:6d} rows")
        else:
            print(f"  ✅ FINAL  {final}")

In [ ]:
# Smoke-test the pipeline with 50 rows per source (~30 sec, ~$0.10)
# Run this on first execution to verify keys work. Skip on subsequent runs.

import importlib
import scripts.build_finetune_corpus as cb
importlib.reload(cb)

DO_DRY_RUN = not (CORPUS_DIR / "v1_train.jsonl").exists()  # only on first run

if DO_DRY_RUN:
    await cb.run(stages=[1, 2, 3, 4, 5, 6], dry_run=True)
else:
    print("✅ Final corpus already exists — skipping dry-run")

In [ ]:
# ☢️ FULL CORPUS BUILD — only runs if v1_train.jsonl isn't already on Drive
import shutil, importlib

if (CORPUS_DIR / "v1_train.jsonl").exists():
    n = sum(1 for _ in (CORPUS_DIR / "v1_train.jsonl").open())
    print(f"✅ Corpus already built ({n} train rows on Drive). Skipping full build.")
    print("   To force rebuild: shutil.rmtree(CORPUS_DIR) then re-run this cell.")
else:
    # Wipe dry-run stage outputs first if they exist (so we re-run on real data)
    stages_dir = CORPUS_DIR / "stages"
    if stages_dir.exists():
        # Detect if these are dry-run stubs by checking row count
        first_stage = stages_dir / "stage1_jumia_filtered.jsonl"
        if first_stage.exists() and sum(1 for _ in first_stage.open()) < 500:
            print("  Wiping dry-run stubs from stages/...")
            shutil.rmtree(stages_dir)

    importlib.reload(cb)
    print("🚀 Building full corpus (resumable; ctrl-C is safe — just re-run this cell)")
    await cb.run(stages=[1, 2, 3, 4, 5, 6], dry_run=False)
    print("\n✅ Corpus build complete")

In [ ]:
# Inspect the final corpus
import json
from collections import Counter

def _stats(p):
    rows = [json.loads(l) for l in p.open()]
    return {
        "n": len(rows),
        "by_register": dict(Counter(r["input"]["register_tier"] for r in rows)),
        "by_source":   dict(Counter(r["_meta"]["source"] for r in rows)),
        "by_rating":   dict(sorted(Counter(r["output"]["rating"] for r in rows).items())),
    }

for split in ["v1_train.jsonl", "v1_val.jsonl", "v1_test.jsonl"]:
    p = CORPUS_DIR / split
    if p.exists():
        print(f"\n=== {split} ===")
        print(json.dumps(_stats(p), indent=2, ensure_ascii=False))

# Show 3 random training examples
import random; random.seed(0)
all_rows = [json.loads(l) for l in (CORPUS_DIR / "v1_train.jsonl").open()]
print(f"\n=== 3 random training examples ===")
for r in random.sample(all_rows, 3):
    print(f"\n---  source: {r['_meta']['source']}  |  rating: {r['output']['rating']}/5  |  register: {r['input']['register_tier']}")
    print(r["output"]["review"][:400])

## Part 2 — QLoRA Fine-Tune NaijaReviewer-8B

**Base**: `meta-llama/Meta-Llama-3.1-8B-Instruct`
**Method**: QLoRA (4-bit NF4) `r=16, α=32`
**Hyperparams**: lifted from `finetuning/configs/naija_reviewer_qlora.yaml`, then GPU-autotuned overrides applied
**Checkpoints**: written to `CHECKPOINT_DIR` on Drive every 200 steps; auto-resumes if interrupted
**Eval**: every 200 steps on the val split

If the Colab runtime disconnects mid-training, just re-run this notebook from the top — Part 1 will skip (corpus exists on Drive) and Part 2 will resume from the last checkpoint.

In [ ]:
# HuggingFace login (required to pull Llama 3.1 Instruct)
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))
print("✅ HuggingFace login OK")
print("   If Llama 3.1 8B Instruct download fails: visit")
print("   https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct and accept the license")

In [ ]:
# Load training config + apply GPU autotune overrides
import yaml, json

with open("finetuning/configs/naija_reviewer_qlora.yaml") as f:
    cfg = yaml.safe_load(f)

# Override with autotuned values
cfg["per_device_train_batch_size"] = GPU_CFG["batch"]
cfg["per_device_eval_batch_size"] = GPU_CFG["batch"]
cfg["gradient_accumulation_steps"] = GPU_CFG["grad_accum"]
cfg["max_seq_length"] = GPU_CFG["seq_len"]
cfg["bf16"] = BF16_OK
cfg["fp16"] = not BF16_OK  # fallback for T4
cfg["output_dir"] = str(CHECKPOINT_DIR)
cfg["dataloader_num_workers"] = GPU_CFG["workers"]
cfg["dataloader_pin_memory"] = True
# Save every 200 steps for resilience; keep 3 most-recent to limit Drive bloat
cfg["save_steps"] = 200
cfg["eval_steps"] = 200
cfg["save_strategy"] = "steps"
cfg["evaluation_strategy"] = "steps"
cfg["save_total_limit"] = 3
cfg["logging_steps"] = 25

print("=== Training config (autotuned) ===")
print(f"  base_model:       {cfg['base_model']}")
print(f"  batch (per dev):  {cfg['per_device_train_batch_size']}")
print(f"  grad_accum:       {cfg['gradient_accumulation_steps']}")
print(f"  effective batch:  {cfg['per_device_train_batch_size'] * cfg['gradient_accumulation_steps']}")
print(f"  max_seq_length:   {cfg['max_seq_length']}")
print(f"  bf16 / fp16:      {cfg['bf16']} / {cfg['fp16']}")
print(f"  epochs:           {cfg['num_train_epochs']}")
print(f"  save_steps:       {cfg['save_steps']}")
print(f"  output_dir:       {cfg['output_dir']}")

# Single SFT formatter — used by both training and eval inference for prompt consistency
def format_example(example):
    inp = example.get("input", {})
    out = example.get("output", {})
    persona = json.dumps(inp.get("persona", {}), ensure_ascii=False)
    product = json.dumps(inp.get("product", {}), ensure_ascii=False)
    register = inp.get("register_tier", "nigerian_english")
    rating = out.get("rating", 3)
    review = (out.get("review", "") or "").replace('"', '\\"')
    return (
        f"### Instruction\n{example.get('instruction','')}\n\n"
        f"### Persona\n{persona}\n\n"
        f"### Product\n{product}\n\n"
        f"### Register tier\n{register}\n\n"
        f"### Response\n"
        f'{{"rating": {rating}, "review": "{review}"}}'
    )

In [ ]:
# Detect existing checkpoint for resume
checkpoints = sorted(CHECKPOINT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
RESUME_FROM = checkpoints[-1] if checkpoints else None

if RESUME_FROM:
    print(f"✅ Found existing checkpoint: {RESUME_FROM.name}")
    print("   Training will RESUME from this checkpoint.")
else:
    print("  No existing checkpoint — training will start from scratch.")

In [ ]:
# Load Llama 3.1 8B in 4-bit + apply LoRA
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type=cfg["bnb_4bit_quant_type"],
    bnb_4bit_compute_dtype=torch.bfloat16 if BF16_OK else torch.float16,
    bnb_4bit_use_double_quant=cfg["bnb_4bit_use_double_quant"],
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(cfg["base_model"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading base model in 4-bit (~1-2 min)...")
model = AutoModelForCausalLM.from_pretrained(
    cfg["base_model"],
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=GPU_CFG["attn"],
    torch_dtype=torch.bfloat16 if BF16_OK else torch.float16,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=cfg["gradient_checkpointing"])

# If resuming, load existing adapter on top of the base; otherwise create fresh
if RESUME_FROM:
    print(f"Loading adapter from checkpoint {RESUME_FROM.name}...")
    model = PeftModel.from_pretrained(model, str(RESUME_FROM), is_trainable=True)
else:
    lora_config = LoraConfig(
        r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"],
        lora_dropout=cfg["lora_dropout"],
        target_modules=cfg["lora_target_modules"],
        task_type="CAUSAL_LM",
        bias="none",
    )
    model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n✅ Model ready")
print(f"  Trainable params: {trainable/1e6:.1f}M  ({100*trainable/total:.2f}% of base)")
print(f"  Attention impl:   {model.config._attn_implementation}")

In [ ]:
# Load training data — already on Drive from Part 1
from datasets import Dataset

ds_train_raw = [json.loads(l) for l in (CORPUS_DIR / "v1_train.jsonl").open()]
ds_val_raw = [json.loads(l) for l in (CORPUS_DIR / "v1_val.jsonl").open()]

ds_train = Dataset.from_list(ds_train_raw).map(lambda r: {"text": format_example(r)}, num_proc=2)
ds_val   = Dataset.from_list(ds_val_raw).map(lambda r: {"text": format_example(r)}, num_proc=2)

print(f"Train: {len(ds_train):,}  Val: {len(ds_val):,}")

# Estimate training time
steps_per_epoch = len(ds_train) // (cfg["per_device_train_batch_size"] * cfg["gradient_accumulation_steps"])
total_steps = steps_per_epoch * cfg["num_train_epochs"]
# Rough sec-per-step estimates (very approximate, vary by GPU)
sec_per_step_estimate = {"h100": 1.2, "a100": 2.0, "l4": 6.0, "t4": 12.0}
matched = next((v for k, v in sec_per_step_estimate.items() if k in GPU_NAME.lower()), 5.0)
estimated_min = total_steps * matched / 60
print(f"Steps/epoch:    {steps_per_epoch}")
print(f"Total steps:    {total_steps}")
print(f"Estimated time: ~{estimated_min:.0f} minutes  ({estimated_min/60:.1f} hours)")

In [ ]:
# Init W&B (optional)
import wandb
try:
    wandb.init(
        project="naija-persona-agent",
        name=cfg["run_name"],
        config={**cfg, "gpu": GPU_NAME, "vram_gb": GPU_VRAM, "flash_attn": FLASH_OK},
        mode="online",
        resume="allow",
    )
    print("✅ W&B initialised")
except Exception as e:
    print(f"⚠️ W&B disabled: {e}")
    os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# Configure SFTTrainer with Drive-persisted checkpoints + resume
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=cfg["output_dir"],
    num_train_epochs=cfg["num_train_epochs"],
    per_device_train_batch_size=cfg["per_device_train_batch_size"],
    per_device_eval_batch_size=cfg["per_device_eval_batch_size"],
    gradient_accumulation_steps=cfg["gradient_accumulation_steps"],
    learning_rate=cfg["learning_rate"],
    lr_scheduler_type=cfg["lr_scheduler_type"],
    warmup_steps=cfg["warmup_steps"],
    weight_decay=cfg["weight_decay"],
    bf16=cfg["bf16"],
    fp16=cfg["fp16"],
    optim=cfg["optim"],
    gradient_checkpointing=cfg["gradient_checkpointing"],
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=cfg["logging_steps"],
    save_strategy=cfg["save_strategy"],
    save_steps=cfg["save_steps"],
    eval_strategy=cfg["evaluation_strategy"],
    eval_steps=cfg["eval_steps"],
    save_total_limit=cfg["save_total_limit"],
    load_best_model_at_end=False,  # not strictly needed with checkpoint resume
    seed=cfg["seed"],
    report_to=["wandb"] if not os.environ.get("WANDB_DISABLED") else ["none"],
    run_name=cfg["run_name"],
    max_length=cfg["max_seq_length"],
    packing=False,
    dataset_text_field="text",
    dataloader_num_workers=cfg["dataloader_num_workers"],
    dataloader_pin_memory=cfg["dataloader_pin_memory"],
    # Frequent saves to Drive — survives Colab disconnect
    save_safetensors=True,
    overwrite_output_dir=False,
    # Performance
    tf32=True,  # only effective on A100/H100; ignored elsewhere
    group_by_length=True,  # batches sequences of similar length together — fewer pad tokens
    report_to=["wandb"] if not os.environ.get("WANDB_DISABLED") else ["none"],
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    tokenizer=tokenizer,
    args=training_args,
)

print("✅ Trainer configured")

In [ ]:
# 🚀 TRAIN — resumes from the last checkpoint if one exists
import time

print("🚀 Starting / resuming training")
print(f"   Checkpoint dir: {cfg['output_dir']}")
print(f"   Saves every {cfg['save_steps']} steps (Drive write)\n")

t0 = time.time()

if RESUME_FROM:
    trainer.train(resume_from_checkpoint=str(RESUME_FROM))
else:
    trainer.train()

elapsed = (time.time() - t0) / 60
print(f"\n✅ Training done in {elapsed:.1f} min")

# Final save of the LoRA adapter
FINAL_ADAPTER = CHECKPOINT_DIR / "final-adapter"
trainer.save_model(str(FINAL_ADAPTER))
tokenizer.save_pretrained(str(FINAL_ADAPTER))
print(f"✅ Final adapter saved → {FINAL_ADAPTER}")

In [ ]:
# Merge LoRA into base for shipping (~5 min, ~16 GB on Drive)
if (MERGED_DIR / "config.json").exists():
    print(f"✅ Merged model already exists at {MERGED_DIR} — skipping merge")
else:
    print("Loading base in bf16 for merge...")
    base = AutoModelForCausalLM.from_pretrained(
        cfg["base_model"],
        torch_dtype=torch.bfloat16 if BF16_OK else torch.float16,
        device_map="auto",
    )
    print("Applying adapter + merging...")
    merged = PeftModel.from_pretrained(base, str(FINAL_ADAPTER))
    merged = merged.merge_and_unload()

    print(f"Saving merged model to {MERGED_DIR}...")
    merged.save_pretrained(str(MERGED_DIR), safe_serialization=True, max_shard_size="5GB")
    tokenizer.save_pretrained(str(MERGED_DIR))
    print("✅ Merged NaijaReviewer-8B saved to Drive")

    # Free up VRAM before eval
    del base, merged
    import gc; gc.collect(); torch.cuda.empty_cache()

## Part 3 — Head-to-Head Evaluation

Run on the held-out `v1_test.jsonl`. Compares NaijaReviewer-8B against:
1. Base Llama 3.1 8B Instruct (isolates the fine-tune contribution)
2. Vanilla Claude Sonnet 4 (frontier closed baseline)
3. Vanilla Nemotron 70B (frontier open-weight baseline)

Metrics: rating RMSE, register-tier match, cultural-marker recall, length ratio. Results saved to Drive.

In [ ]:
# Load the merged model for inference (frees adapter overhead)
from transformers import pipeline

ds_test = [json.loads(l) for l in (CORPUS_DIR / "v1_test.jsonl").open()]
print(f"Test set: {len(ds_test)} rows")

naija_model = AutoModelForCausalLM.from_pretrained(
    str(MERGED_DIR),
    torch_dtype=torch.bfloat16 if BF16_OK else torch.float16,
    device_map="auto",
)
naija_pipe = pipeline(
    "text-generation",
    model=naija_model,
    tokenizer=tokenizer,
    device_map="auto",
)

import re

def naija_predict(example, max_new_tokens=300):
    prompt = format_example(example).rsplit("### Response\n", 1)[0] + "### Response\n"
    out = naija_pipe(prompt, max_new_tokens=max_new_tokens,
                     temperature=0.7, do_sample=True,
                     pad_token_id=tokenizer.eos_token_id, return_full_text=False)
    text = out[0]["generated_text"].strip()
    m = re.search(r'\{[\s\S]*?\}', text)
    if not m:
        return None, text
    try:
        parsed = json.loads(m.group())
        return int(parsed.get("rating", 3)), str(parsed.get("review", ""))
    except Exception:
        return None, text

# Smoke-test
ex = ds_test[0]
rating, review = naija_predict(ex)
print(f"\nGT:   rating={ex['output']['rating']} review={ex['output']['review'][:120]}")
print(f"\nNaija: rating={rating}\n        review={review[:300]}")

In [ ]:
# Eval helpers
import statistics

PIDGIN_MARKERS = {"abeg", "wahala", "no cap", "biko", "nna", "scatter", "shey", "sef",
                  "haba", "wallahi", "omo", "naija", "no shaking", "sharp sharp",
                  "dem", "wetin", "e clear", "e too much", "e dey"}

def detect_register_naive(text):
    t = text.lower()
    hits = sum(1 for m in PIDGIN_MARKERS if m in t)
    if hits >= 3: return "nigerian_pidgin"
    if hits >= 1: return "code_mixed"
    if any(m in t for m in ["well done sir", "thank God", "by God"]):
        return "nigerian_english"
    return "standard_english"

def cultural_marker_recall(pred, gt):
    gt_markers = {m for m in PIDGIN_MARKERS if m in gt.lower()}
    if not gt_markers: return None
    pred_l = pred.lower()
    return sum(1 for m in gt_markers if m in pred_l) / len(gt_markers)

def compute_metrics(preds, gts):
    rmse_sq, reg_match, cmr, len_ratio, valid = [], 0, [], [], 0
    for (pr, pt), gt in zip(preds, gts):
        if pr is None: continue
        valid += 1
        rmse_sq.append((pr - gt["output"]["rating"]) ** 2)
        if detect_register_naive(pt) == gt["input"]["register_tier"]: reg_match += 1
        s = cultural_marker_recall(pt, gt["output"]["review"])
        if s is not None: cmr.append(s)
        if gt["output"]["review"]:
            len_ratio.append(len(pt) / max(len(gt["output"]["review"]), 1))
    return {
        "n_valid": valid,
        "RMSE": (statistics.mean(rmse_sq) ** 0.5) if rmse_sq else None,
        "register_match_pct": 100 * reg_match / max(valid, 1),
        "cultural_marker_recall": statistics.mean(cmr) if cmr else None,
        "length_ratio_mean": statistics.mean(len_ratio) if len_ratio else None,
    }

print("✅ Eval helpers loaded")

In [ ]:
# Run NaijaReviewer-8B on a 100-row eval subset
from tqdm.notebook import tqdm

EVAL_N = 100  # bump to len(ds_test) for paper run
eval_subset = ds_test[:EVAL_N]

print(f"Running NaijaReviewer-8B on {len(eval_subset)} rows...")
naija_preds = [naija_predict(ex) for ex in tqdm(eval_subset)]
naija_metrics = compute_metrics(naija_preds, eval_subset)

print("\n=== NaijaReviewer-8B ===")
print(json.dumps(naija_metrics, indent=2))

In [ ]:
# Vanilla baselines via API (async + parallel)
import asyncio

BASELINE_SYSTEM = (
    "You are a Nigerian product reviewer. Generate a review and a 1-5 star rating that "
    "matches the user's persona — their register, intensity, and aspects they care about. "
    "Output STRICT JSON: {\"rating\": <int 1-5>, \"review\": \"<text>\"}."
)

async def baseline_predict(client, example, sem):
    async with sem:
        prompt = format_example(example).rsplit("### Response\n", 1)[0]
        try:
            r = await client.complete_json(prompt=prompt, system=BASELINE_SYSTEM, max_tokens=300, temperature=0.7)
            return int(r.get("rating", 3)), str(r.get("review", ""))
        except Exception:
            return None, ""

async def run_baseline(client, examples):
    sem = asyncio.Semaphore(8)
    tasks = [baseline_predict(client, ex, sem) for ex in examples]
    out = []
    for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
        out.append(await fut)
    return out

results = {"naija_reviewer_8b": naija_metrics}

print("\n=== Vanilla Claude Sonnet 4 ===")
cl = asyncio.run(run_baseline(LLMClient("anthropic:claude-sonnet-4-20250514"), eval_subset))
results["vanilla_claude_sonnet_4"] = compute_metrics(cl, eval_subset)
print(json.dumps(results["vanilla_claude_sonnet_4"], indent=2))

print("\n=== Vanilla Nemotron 70B ===")
nv = asyncio.run(run_baseline(LLMClient("nvidia:nvidia/llama-3.1-nemotron-70b-instruct"), eval_subset))
results["vanilla_nemotron_70b"] = compute_metrics(nv, eval_subset)
print(json.dumps(results["vanilla_nemotron_70b"], indent=2))

In [ ]:
# Save results to Drive + repo paper/ dir + pretty table
(RESULTS_DIR / "results.json").write_text(json.dumps(results, indent=2))
Path("paper").mkdir(exist_ok=True)
Path("paper/results.json").write_text(json.dumps(results, indent=2))

print("\n" + "=" * 105)
print(f"{'Model':<35} {'RMSE↓':>8} {'Register↑':>12} {'Marker recall↑':>16} {'Length ratio':>14}")
print("=" * 105)
for name, m in results.items():
    print(f"{name:<35} "
          f"{m.get('RMSE') or 0:>8.3f} "
          f"{m.get('register_match_pct') or 0:>11.1f}% "
          f"{(m.get('cultural_marker_recall') or 0)*100:>15.1f}% "
          f"{(m.get('length_ratio_mean') or 0):>14.2f}")
print("=" * 105)
print(f"\n✅ Saved to {RESULTS_DIR / 'results.json'} and paper/results.json")

## Part 4 — Push to HuggingFace

Publishes:
- Merged NaijaReviewer-8B weights (sharded ≤5GB safetensors)
- LoRA adapter (~150 MB)
- Fine-tune corpus dataset
- Model card with eval table, training recipe, bias notes, citation

Update `HF_ORG` to your HuggingFace org/username.

In [ ]:
from huggingface_hub import HfApi, create_repo, upload_folder

HF_ORG = "Mystique1337"  # ← change to your HF org/user
MODEL_REPO   = f"{HF_ORG}/naija-reviewer-8b"
ADAPTER_REPO = f"{HF_ORG}/naija-reviewer-8b-lora"
DATASET_REPO = f"{HF_ORG}/npa-corpus-v1"

for repo, kind in [(MODEL_REPO, "model"), (ADAPTER_REPO, "model"), (DATASET_REPO, "dataset")]:
    try:
        create_repo(repo, repo_type=kind, exist_ok=True, private=True)
        print(f"  ✅ {kind:7s}: {repo}")
    except Exception as e:
        print(f"  ⚠️ {kind:7s}: {e}")

In [ ]:
# Generate model card
summary_path = CORPUS_DIR / "v1_summary.json"
n_train = json.loads(summary_path.read_text()).get("train", {}).get("n", "?") if summary_path.exists() else "?"

card = f"""---
language: [en, pcm, ha, ig, yo]
license: llama3.1
library_name: transformers
tags: [nigerian, pidgin, review-generation, persona-conditioned, qlora, llama-3.1]
datasets: [{DATASET_REPO}]
base_model: meta-llama/Meta-Llama-3.1-8B-Instruct
---

# NaijaReviewer-8B

QLoRA fine-tune of Llama 3.1 8B Instruct on ~{n_train} Nigerian product review examples. Hybrid corpus:
real Jumia reviews + AfriSenti Pidgin reformulations + persona-conditioned synthetic.

**Submission to the Nigerian AI Agents Hackathon, May 2026** — Naija Persona Agent.

## Intended use
Persona-conditioned generation of authentic Nigerian product reviews. Given persona JSON + product JSON + register tier, outputs a 1-5 star rating + review text in the user's voice.

## Training
- **Base**: `meta-llama/Meta-Llama-3.1-8B-Instruct`
- **Method**: QLoRA (4-bit NF4), `r=16, α=32`, target attention + MLP projections
- **Corpus**: {DATASET_REPO}
- **Hyperparams**: lr=2e-4 cosine, effective batch=32, epochs=2, bf16, seed=42
- **Hardware**: {GPU_NAME} ({GPU_VRAM:.0f} GB), flash_attn={FLASH_OK}

## Eval (held-out test split of npa-corpus-v1)

| Model | RMSE ↓ | Register match ↑ | Cultural-marker recall ↑ |
|---|---|---|---|
| **NaijaReviewer-8B (ours)** | {results['naija_reviewer_8b'].get('RMSE', 0) or 0:.3f} | {results['naija_reviewer_8b'].get('register_match_pct', 0):.1f}% | {(results['naija_reviewer_8b'].get('cultural_marker_recall') or 0)*100:.1f}% |
| Vanilla Claude Sonnet 4 | {results.get('vanilla_claude_sonnet_4',{}).get('RMSE', 0) or 0:.3f} | {results.get('vanilla_claude_sonnet_4',{}).get('register_match_pct', 0):.1f}% | {(results.get('vanilla_claude_sonnet_4',{}).get('cultural_marker_recall') or 0)*100:.1f}% |
| Vanilla Nemotron 70B | {results.get('vanilla_nemotron_70b',{}).get('RMSE', 0) or 0:.3f} | {results.get('vanilla_nemotron_70b',{}).get('register_match_pct', 0):.1f}% | {(results.get('vanilla_nemotron_70b',{}).get('cultural_marker_recall') or 0)*100:.1f}% |

## Bias and limitations
- Trained primarily on Jumia scrape + AfriSenti tweets; not validated against full diversity of Nigerian dialects.
- See \"Lost in Simulation\" (arXiv:2601.17087) — LLM-simulated users diverge from real users on action sequences.
- Inherits Llama 3.1 base biases.

## Citation
```bibtex
@misc{{npa2026,
  title={{Naija Persona Agent: A Cultural-Prior-Aware LLM Agent for Nigerian Review Simulation and Recommendation}},
  author={{Ashinze, Franca, and team}},
  year={{2026}},
  url={{https://github.com/{HF_ORG}/telcoproject}}
}}
```

## License
Llama 3.1 Community License. Inputs include AfriSenti (CC-BY 4.0) and a public Jumia review scrape (attributed). See dataset card for full attribution.
"""

(MERGED_DIR / "README.md").write_text(card)
print("✅ Model card written")

In [ ]:
# Upload
print(f"Uploading merged model → {MODEL_REPO}")
upload_folder(folder_path=str(MERGED_DIR), repo_id=MODEL_REPO, repo_type="model")
print(f"  ✅ https://huggingface.co/{MODEL_REPO}")

print(f"\nUploading LoRA adapter → {ADAPTER_REPO}")
upload_folder(
    folder_path=str(FINAL_ADAPTER),
    repo_id=ADAPTER_REPO,
    repo_type="model",
    ignore_patterns=["merged/*", "checkpoint-*", "runs/*", "global_step*"],
)
print(f"  ✅ https://huggingface.co/{ADAPTER_REPO}")

print(f"\nUploading corpus → {DATASET_REPO}")
upload_folder(
    folder_path=str(CORPUS_DIR),
    repo_id=DATASET_REPO,
    repo_type="dataset",
    ignore_patterns=["raw/*", "stages/*"],
)
print(f"  ✅ https://huggingface.co/datasets/{DATASET_REPO}")

print("\n🎉 All artifacts on HuggingFace (private — flip to public when ready for the hackathon submission)")

## Part 5 — GGUF Conversion for Ollama (local step)

Inside the FastAPI container we serve NaijaReviewer-8B via local Ollama. After this notebook finishes, on your local machine:

```bash
git clone https://github.com/ggerganov/llama.cpp && cd llama.cpp && make

# Pull merged model from HF
huggingface-cli download Mystique1337/naija-reviewer-8b --local-dir ./naija-reviewer-merged

# Convert and quantize
python3 convert_hf_to_gguf.py ./naija-reviewer-merged --outfile naija-reviewer-8b-f16.gguf --outtype f16
./llama-quantize naija-reviewer-8b-f16.gguf naija-reviewer-8b-Q4_K_M.gguf Q4_K_M

# Register with Ollama
cp naija-reviewer-8b-Q4_K_M.gguf telcoproject/finetuning/
cd telcoproject && ollama create naija-reviewer-8b -f finetuning/Modelfile
ollama run naija-reviewer-8b  # smoke-test
```

Then flip the FastAPI backbone in `.env`:

```bash
TASK1_BACKBONE=ollama:naija-reviewer-8b
```

Restart `make demo` — the container now serves NaijaReviewer-8B locally with zero API cost.

---

## Done. Output of this notebook (all on Drive):

- ✅ `~/Drive/MyDrive/naija-persona-agent/data/finetune/v1_*.jsonl` — ~20k training corpus
- ✅ `~/Drive/MyDrive/naija-persona-agent/checkpoints/naija-reviewer-v1/` — training checkpoints + final adapter
- ✅ `~/Drive/MyDrive/naija-persona-agent/merged/naija-reviewer-8b/` — merged shippable model
- ✅ `~/Drive/MyDrive/naija-persona-agent/results/results.json` — eval table for the paper
- ✅ HuggingFace: `naija-reviewer-8b` model + adapter + corpus dataset (private until you publish)

**If Colab disconnects, re-run the notebook from the top.** Part 1 skips (corpus on Drive), Part 2 resumes from the last checkpoint, Part 3 re-runs eval if needed, Part 4 re-pushes safely.